[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/06_%E7%99%BA%E5%B1%95_%E3%83%9E%E3%83%BC%E3%82%B1%E5%88%86%E6%9E%90/02_%E6%99%82%E7%B3%BB%E5%88%97%E4%BA%88%E6%B8%AC.ipynb)

# 発展マーケ-02. 時系列予測（トレンド・季節性・指数平滑）

> 🟢 Colab（ブラウザ）で実行できます。**最初に「① セットアップ」セルを実行**してください。
> 📌 **発展編（統計検定2級の先：準1級〜実務/データサイエンス）**。scikit-learn・statsmodels を使います（Colabは導入済み。ローカルは `uv add scikit-learn statsmodels`）。

36か月分の売上から、**来月以降を予測**します。移動平均→線形トレンド→**Holt-Winters（指数平滑）** と段階的に。

In [ ]:
# === ① セットアップ（最初に実行してください）===
import pandas as pd               # 表データ
import numpy as np                # 数値計算
import matplotlib.pyplot as plt   # グラフ描画
import os
plt.rcParams['axes.unicode_minus'] = False   # マイナス記号の文字化け防止
# ローカルに無ければ公開リポジトリ(raw)から読み込む共通関数
RAW = 'https://raw.githubusercontent.com/Carlo-Broschi/statistics-python-for-students/main/data/'
def load(name):
    p = f'../data/{name}'
    return pd.read_csv(p) if os.path.exists(p) else pd.read_csv(RAW + name)
kpi = load('monthly_kpi.csv')   # 月次KPI（売上・新規リード・受注数）
kpi.head()

## 1. まず眺める（トレンド＋季節性）

In [ ]:
kpi['売上'].plot(marker='o', figsize=(10,4), title='月別売上（36か月）')  # 折れ線
plt.ylabel('売上(万円)'); plt.show()
print('右肩上がりのトレンド＋年末(12月)が高い季節性が見える')

## 2. 移動平均でならす
12か月移動平均で季節の波を消し、大きな流れ（トレンド）を見ます。

In [ ]:
kpi['移動平均12'] = kpi['売上'].rolling(12).mean()   # 12か月移動平均
kpi[['売上','移動平均12']].plot(figsize=(10,4)); plt.ylabel('売上'); plt.show()

## 3. 線形トレンドで予測（時間で回帰）
「月の番号 t」で売上を回帰し、直線を伸ばして予測します。

In [ ]:
t = np.arange(len(kpi))                      # 0,1,2,... の月番号
a, b = np.polyfit(t, kpi['売上'], 1)          # 最小二乗で 傾きa・切片b
print(f'トレンド: 売上 ≈ {a:.1f}×月 + {b:.0f}')
future = np.arange(len(kpi), len(kpi)+3)     # 次の3か月
print('単純トレンド予測:', (a*future + b).round().astype(int))
print('→ 直線だけだと季節性（年末の山）を無視してしまう')

## 4. Holt-Winters（指数平滑：トレンド＋季節性）
**最近のデータを重視**しつつ、トレンドと季節性を同時に学習する定番手法です。

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing
ts = kpi['売上'].astype(float)               # 予測対象の系列
model = ExponentialSmoothing(ts, trend='add', seasonal='add',
                             seasonal_periods=12).fit()   # 加法トレンド＋12か月季節
fc = model.forecast(3)                       # 来3か月を予測
print('来3か月の予測:', fc.round().astype(int).tolist())

ts.plot(label='実績')                                    # 実績
model.fittedvalues.plot(label='あてはめ')                # モデルの再現
pd.Series(fc.values, index=range(len(ts), len(ts)+3)).plot(
          label='予測', marker='o')                       # 予測
plt.legend(); plt.title('Holt-Wintersによる予測'); plt.show()

季節性を反映し、年末の山も織り込んだ予測になります。

## 5. 予測の精度を測る（学習/検証に分ける）
最後の6か月を「答え合わせ用」に取り分け、予測のズレを測ります。

In [ ]:
train, test = ts[:-6], ts[-6:]               # 学習用と検証用に分割
m2 = ExponentialSmoothing(train, trend='add', seasonal='add', seasonal_periods=12).fit()
pred = m2.forecast(6)                         # 検証期間を予測
mae = np.abs(pred.values - test.values).mean()                 # 平均絶対誤差
mape = (np.abs(pred.values - test.values) / test.values).mean() * 100  # 平均絶対%誤差
print(f'MAE = {mae:.1f} 万円,  MAPE = {mape:.1f}%')
print('→ MAPEが小さいほど予測が当たっている')

---
## 🏆 練習問題

**問1.** `受注数` を同じ手順で予測しよう。

**問2.** `seasonal='mul'`（乗法季節）や `trend=None` に変え、MAPEがどう変わるか比べよう。

**問3.** 移動平均を3か月と12か月で重ね、見え方の違いを確かめよう。

**問4.** 検証期間を最後の12か月に変えてMAPEを測り直そう。

In [ ]:
# 問1〜4


> 🔑 **解答例は別ページ**（ネタバレ防止）👉 **[解答例を開く](https://github.com/Carlo-Broschi/statistics-python-for-students/blob/main/%E8%A7%A3%E7%AD%94%E9%9B%86/06_%E7%99%BA%E5%B1%95_%E3%83%9E%E3%83%BC%E3%82%B1%E5%88%86%E6%9E%90/02_%E6%99%82%E7%B3%BB%E5%88%97%E4%BA%88%E6%B8%AC.md)**